In [1]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 26.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [35]:
import pandas as pd
import numpy as np
import torch
import transformers
from bertopic import BERTopic
import os
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
from transformers import AutoTokenizer, pipeline
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import TextGeneration
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt
import openai
import lightgbm as lgb
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.decomposition import PCA
import nltk
import re
import plotly.io as pio
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [5]:
# os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
# df1 = pd.read_csv('../data/letters_2021.csv')
# df2 = pd.read_csv('../data/sentence_sets_trimmed.csv', encoding='mac-roman')
# df2 = df2[df2['full_text_tokens'] > 10]

# Degender Data

In [6]:
# raw_pairs = {}
# with open("../data/gendered_term_list.txt", "r", encoding="utf-8") as f:
#     for line in f:
#         key, value = line.strip().split("\t")
#         raw_pairs[key] = value

In [7]:
# degender_mapping = {
#     rf"(?:^|\b|[^\w\s])(?P<token>{re.escape(k)})(?P<suffix>'s|’s)?(?=\b|[^\w\s]|$)": v
#     for k, v in raw_pairs.items()
# }

### First Dataset

In [8]:
# df1 = df1[['LETTERTEXT', 'LETTER_GENDER']]
# df1 = df1.rename(columns={'LETTERTEXT':'full_text', 'LETTER_GENDER':'label'})
# df1['full_text'] = df1['full_text'].str.lower()

In [9]:
# def apply_degendering(text):
#     for pattern, replacement in degender_mapping.items():
#         def repl(match):
#             token = match.group("token")
#             suffix = match.group("suffix") or ""
#             replacement_base = raw_pairs.get(token.lower(), token)
#             return match.group(0).replace(token + suffix, replacement_base + suffix)
#         text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
#     return text

In [10]:
# df1['full_text'] = df1['full_text'].apply(apply_degendering)

### Second Dataset

In [11]:
# df2 = df2[['TEXT', 'applicant_gender']]
# df2 = df2.rename(columns={'applicant_gender':'label', 'TEXT':'full_text'})
# df2['full_text'] = df2['full_text'].str.lower()

In [12]:
# df2['full_text'] = df2['full_text'].apply(apply_degendering)

### Combine Datasets

In [13]:
# df = pd.concat([df1, df2], ignore_index=True)

In [14]:
# gender_label_mapping = {
#     'F':0,
#     'female':0,
#     'M':1,
#     'male':1
# }

In [15]:
# df['label'] = df['label'].replace(gender_label_mapping)

In [16]:
# df.to_csv('../data/combined_letters_degendered_topic_modeling.csv', index=False)

In [17]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df = pd.read_csv('../data/combined_letters_degendered_topic_modeling.csv')

# Process Data

In [18]:
letters = df['full_text']
sentences = [sent_tokenize(letter) for letter in letters]
sentences = pd.concat([df, pd.Series(sentences, name='sentences')], axis=1)
sentences = sentences[['sentences']].explode('sentences')
df_sentences = pd.merge(df, sentences, left_index=True, right_index=True)

In [19]:
df_sentences

,full_text,label,sentences
0,it is my pleasure to write a letter of recomme...,0,it is my pleasure to write a letter of recomme...
0,it is my pleasure to write a letter of recomme...,0,her pleasing personality and sincere dedicatio...
0,it is my pleasure to write a letter of recomme...,0,i first met identifier during her hospice and ...
0,it is my pleasure to write a letter of recomme...,0,"as part of the team, identifier took part in e..."
0,it is my pleasure to write a letter of recomme...,0,"prior to seeing patients, she diligently read ..."
...,...,...,...
8986,please accept my strongest recommendation for ...,1,"furthermore , she has a strong research backgr..."
8986,please accept my strongest recommendation for ...,1,her work experience is rounded out with numero...
8986,please accept my strongest recommendation for ...,1,"taken together , first_name is first_name exce..."
8986,please accept my strongest recommendation for ...,1,she will be a great asset to any program she m...


# Topic Modeling

In [20]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedding_model.encode(sentences['sentences'].tolist(), show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5345 [00:00<?, ?it/s]

In [21]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=150, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [22]:
# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
pos_model = PartOfSpeech("en_core_web_sm", top_n_words=10)

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# GPT-3.5
client = openai.OpenAI(api_key="sk-...")
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
topic: <topic label>
"""
openai_model = OpenAI(client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    # "OpenAI": openai_model,  # Uncomment if you will use OpenAI
    "MMR": mmr_model,
    "POS": pos_model,
    # "zephyr": zephyr

}

In [23]:
topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True,
)

# Train model
topics, probs = topic_model.fit_transform(df_sentences['sentences'].tolist(), embeddings)



2025-04-18 14:44:35,276 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-04-18 14:51:38,651 - BERTopic - Dimensionality - Completed ✓
2025-04-18 14:51:38,656 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-04-18 14:52:22,467 - BERTopic - Cluster - Completed ✓
2025-04-18 14:52:22,502 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-04-18 14:52:56,567 - BERTopic - Representation - Completed ✓


# Save Model

In [24]:
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
topic_model.save("../saved_models/topic_models/", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

# Analyze Topics

In [25]:
topic_info = topic_model.get_topic_info()

In [26]:
topic_info

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,41900,-1_research_patient_skills_staff,"[research, patient, skills, staff, help, proce...","[patients, clinical, patient care, clinic, med...","[research, skills, staff, procedures, clinical...","[research, patient, skills, staff, procedures,...","[senior partner and ceo , zen anesthesia llc e..."
1,0,46823,0_dr identifier_dr_ms identifier_possible_iden...,"[dr identifier, dr, ms identifier, possible_id...","[patients identifier, identifier medical, dr i...","[dr identifier, ms identifier, possible_identi...","[directress, ms, center, students, hospital, i...",[identifier has demonstrated utmost qualities ...
2,1,25301,1_middle_name_last_name_first_name last_name_f...,"[middle_name, last_name, first_name last_name,...","[residency first_name, last_name applying, pat...","[first_name last_name, support first_name, rec...","[middle_name, last_name, first_name, äù, long ...",[it is with great pleasure for me to write thi...
3,2,4358,2_anesthesiologist_anesthesiology_career anest...,"[anesthesiologist, anesthesiology, career anes...","[future anesthesiologist, anesthesiologist fut...","[career anesthesiology, field anesthesiology, ...","[anesthesiologist, anesthesiology, excellent a...",[there is no question she will make a fantasti...
4,3,4316,3_rapport_families_patients_patients families,"[rapport, families, patients, patients familie...","[compassionate patients, liked patients, care ...","[patients families, rapport patients, compassi...","[rapport, families, patients, empathy, staff, ...",[her compassion and empathy for her patients a...
...,...,...,...,...,...,...,...,...
72,71,179,71_navy_flight_marine_air,"[navy, flight, marine, air, flight surgeon, of...","[squadron medical, awarded navy, navy marine, ...","[navy, flight surgeon, military, squadron, air...","[navy, flight, marine, air, officer, combat, m...",[for her level-headed leadership and efficienc...
73,72,166,72_identifier support_application residency_re...,"[identifier support, application residency, re...","[residency letter, residency writing, residenc...","[identifier support, residency application, su...","[application, support, letter, behalf, recomme...",[it is my pleasure to write this letter of rec...
74,73,162,73_reservation_recommendation reservation_iden...,"[reservation, recommendation reservation, iden...","[recommendation reservation, reservation recom...","[recommendation reservation, applicant reserva...","[reservation, highest recommendation, highest,...",[i give identifier identifier my highest recom...
75,74,160,74_candidate hesitate_regarding excellent_appl...,"[candidate hesitate, regarding excellent, appl...","[excellent applicant, regarding applicant, wor...","[candidate hesitate, regarding excellent, appl...","[excellent applicant, candidate, applicant, in...",[please let us know if we can provide any furt...


In [27]:
topic_model.get_topic(13, full=True)

{'Main': [('questions', np.float64(0.4808978991196214)),
  ('asked', np.float64(0.4455000811045138)),
  ('asks', np.float64(0.42954659317943544)),
  ('thoughtful questions', np.float64(0.39053055722380653)),
  ('questions asked', np.float64(0.35514926876362896)),
  ('answer', np.float64(0.35481259144152333)),
  ('asked thoughtful', np.float64(0.3505422730549299)),
  ('insightful', np.float64(0.3474428035161068)),
  ('insightful questions', np.float64(0.3384646829034763)),
  ('answers', np.float64(0.3278184262471939))],
 'KeyBERT': [('asks thoughtful', np.float32(0.61275405)),
  ('thoughtful questions', np.float32(0.610398)),
  ('asked thoughtful', np.float32(0.576595)),
  ('questions eager', np.float32(0.55375904)),
  ('questions insightful', np.float32(0.53711057)),
  ('insightful questions', np.float32(0.53710127)),
  ('asks questions', np.float32(0.52936137)),
  ('asks insightful', np.float32(0.5290771)),
  ('asked questions', np.float32(0.51616365)),
  ('questions asks', np.float32

In [28]:
topic_labels = topic_model.generate_topic_labels(nr_words=5,
                                                 topic_prefix=True,
                                                 word_length=100,
                                                 separator="||",
                                                 aspect='POS')

In [29]:
topic_model.set_topic_labels(topic_labels)

In [30]:
topic_mapping = dict(zip(topic_model.get_topic_info()['Topic'], topic_model.get_topic_info()['CustomName']))
topic_mapping = {f'topic_{key}': value for key, value in topic_mapping.items()}

In [31]:
df_sentences['topic'] = topics

# Create Topics Dataframe

In [32]:
topics_df = df_sentences.copy()
topics_df['topic'] = topics_df['topic'].astype(str)
topics_df = topics_df.groupby(topics_df.index).agg({'topic':' '.join})
topics_df = pd.DataFrame(topics_df)
topics_df = topics_df['topic'].apply(lambda x:x.split(' '))
topics_df = topics_df.reset_index()
all_topics = set(cat for sublist in topics_df['topic'] for cat in sublist)
for topic in all_topics:
    topics_df[topic] = topics_df['topic'].apply(lambda x: 1 if topic in x else 0)
topics_df = topics_df.drop(columns=['index', 'topic'])
topics_df.columns = 'topic_' + topics_df.columns
df_sentences_collapsed = df_sentences[['full_text', 'label']].drop_duplicates()
topics_df = pd.merge(df_sentences_collapsed, topics_df, left_index=True, right_index=True)

In [34]:
# topics_df.to_csv('../data/combined_letters_degendered_with_topics.csv', index=False)

# Analyze Important Topics

In [36]:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
fig = topic_model.visualize_documents(sentences['sentences'].tolist(), hide_annotations=True, reduced_embeddings=reduced_embeddings)

In [37]:
pio.write_html(fig, file="topics_interactive_plot.html", auto_open=True)